# Tratamento dos dados de Pagamentos e Garantias

**Inputs**\
Planilhas fornecidas pelo ELAW e tratadas pela área finaceira. Pagamentos e Garantias.

**Outputs**\
Uma ~Global View~ tabela (stg_trab_pgto_garantias) com os dados de Pagamentos e Garantias tratados e unidos para utilização pelo passo 3.

### 1- Carga e tratamento inicial das duas bases

In [0]:
from pyspark.sql.functions import coalesce, row_number, lit

# Configura campo para que o usuário insira parâmetros

# Parametro do formato de data do arquivo. Ex: 20240423
dbutils.widgets.text("nmtabela_pgto", "")
dbutils.widgets.text("nmtabela_garantias", "")

In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Caminho das pastas e arquivos
nmtabela_pgto = dbutils.widgets.get("nmtabela_pgto")
nmtabela_garantias = dbutils.widgets.get("nmtabela_garantias")

path_pagamentos = f'/Volumes/databox/juridico_comum/arquivos/bases_pagamentos/HISTORICO_DE-PAGAMENTOS_{nmtabela_pgto}.xlsx'
path_garantias = f'/Volumes/databox/juridico_comum/arquivos/bases_garantias/GARANTIAS_{nmtabela_garantias}.xlsx'

In [0]:
# Carrega as planilhas em Spark Data Frames
df_pgto = read_excel(path_pagamentos, "A6",)
df_garantias = read_excel(path_garantias, "A6")

In [0]:
# Lista as colunas com datas
pgto_data_cols = find_columns_with_word(df_pgto, 'DATA ')
garantias_data_cols = find_columns_with_word(df_garantias, 'DATA ')

print("pgto_data_cols")
print(pgto_data_cols)
print("\n")
print("garantias_data_cols")
print(garantias_data_cols)

pgto_data_cols
['DATA REGISTRADO', 'DATA SOLICITAÇÃO (ESCRITÓRIO)', 'DATA LIMITE DE PAGAMENTO', 'DATA DE VENCIMENTO', 'DATA EFETIVA DO PAGAMENTO']


garantias_data_cols
['DATA REGISTRADO PROCESSO', 'DATA DE DISTRIBUIÇÃO DO PROCESSO', 'DATA DE LEVANTAMENTO (EMPRESA)', 'DATA DE LEVANTAMENTO (PARTE CONTRÁRIA)', 'DATA DE ACIONAMENTO DO FLUXO', 'DATA LIMITE DE PAGAMENTO', 'DATA DE VENCIMENTO', 'DATA EFETIVA DO PAGAMENTO', 'DATA DO ALVARÁ', 'DATA DA CI', 'DATA DA EMISSÃO - RECURSAL', 'DATA DA VIGÊNCIA - RECURSAL', 'DATA REGISTRADO - JUDICIAL', 'DATA DA EMISSÃO - JUDICIAL', 'DATA DA VIGÊNCIA - JUDICIAL', 'DATA DA REATIVAÇÃO']


In [0]:
# Converte as datas das colunas listadas
df_pgto = convert_to_date_format(df_pgto, pgto_data_cols)
df_garantias = convert_to_date_format(df_garantias, garantias_data_cols)

In [0]:
df_garantias.count() # Quantidade de linhas de Garantias


200638

In [0]:
len(df_garantias.columns) # Quantidade de Colunas de Garantias

65

In [0]:
# df_garantias.count() #165887

### 2 - Tratamento base de Pagamentos

In [0]:
df_pgto.createOrReplaceTempView("HISTORICO_DE_PAGAMENTOS_1")

In [0]:
df_pgto = spark.sql("""
SELECT HISTORICO_DE_PAGAMENTOS_1.*,
    CASE 
        WHEN `SUB TIPO` IN ('ACORDO - CÍVEL', 'ACORDO - CÍVEL ESTRATÉGICO', 'ACORDO - CÍVEL MASSA', 'ACORDO - TRABALHISTA', 'PAGAMENTO DE ACORDO - TRABALHISTA (FK)', 'RNO - ACORDO - TRABALHISTA', 'ACORDO - CÍVEL MASSA - CARTÕES', 'ACORDO - CÍVEL MASSA - FORNECEDOR', 'ACORDO - CÍVEL MASSA - MKTPLACE', 'ACORDO - CÍVEL MASSA - SEGURO', 'ACORDO - IMOBILIÁRIO', 'ACORDO - MEDIAÇÃO', 'ACORDO - REGULATÓRIO - PROCON COM AUDIÊNCIA', 'ACORDO/ PROGRAMAS DE PARCELAMENTO - REGULATÓRIO')
        THEN VALOR
    END AS ACORDO,
    CASE 
        WHEN `SUB TIPO` IN ('CONDENAÇÃO - CÍVEL', 'CONDENAÇÃO - CÍVEL ESTRATÉGICO', 'CONDENAÇÃO - CÍVEL MASSA', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO)', 'CONDENAÇÃO - REGULATÓRIO', 'CONDENAÇÃO - TRABALHISTA', 'CONDENAÇÃO - TRABALHISTA (INCONTROVERSO)', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO) - CARTÕES', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO) - FORNECEDOR', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO) - MKTPLACE', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO) - SEGURO', 'CONDENAÇÃO - CÍVEL MASSA - CARTÕES', 'CONDENAÇÃO - CÍVEL MASSA - FORNECEDOR', 'CONDENAÇÃO - CÍVEL MASSA - MKTPLACE', 'CONDENAÇÃO - CÍVEL MASSA - SEGURO', 'CONDENAÇÃO - IMOBILIÁRIO', 'CONDENAÇÃO - IMOBILIÁRIO (INCONTROVERSO)', 'CONDENAÇÃO - REGULATÓRIO', 'MULTA - REGULATÓRIO', 'MULTA PROCESSUAL - CÍVEL', 'MULTA PROCESSUAL - TRABALHISTA', 'MULTA PROCESSUAL - CÍVEL ESTRATÉGICO', 'MULTA PROCESSUAL - IMOBILIÁRIO', 'RNO - CONDENAÇÃO - TRABALHISTA', 'RNO - CONDENAÇÃO - TRABALHISTA (INCONTROVERSO)', 'RNO - MULTA PROCESSUAL - TRABALHISTA')
        THEN VALOR
    END AS CONDENACAO,
    CASE 
        WHEN `SUB TIPO` IN ('HONORÁRIOS CONCILIADOR - CIVEL MASSA', 'HONORÁRIOS PERICIAIS - CÍVEL ESTRATÉGICO', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA', 'HONORÁRIOS PERICIAIS - IMOBILIÁRIO', 'HONORÁRIOS PERICIAIS - TRABALHISTA', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA - CARTÕES', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA - FORNECEDOR', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA - MKTPLACE', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA - SEGURO', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA - CARTÕES', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA - FORNECEDOR', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA - MKTPLACE', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA - SEGURO', 'HONORÁRIOS SUCUMBENCIAIS - REGULATÓRIO', 'PAGAMENTO DE HONORÁRIOS PERICIAIS - TRABALHISTA (FK)', 'RNO - HONORÁRIOS PERICIAIS -TRABALHISTA', 'CUSTAS FK - TRABALHISTA', 'CUSTAS PERITOS - CÍVEL', 'CUSTAS PERITOS - IMOBILIÁRIO', 'CUSTAS PROCESSUAIS - CÍVEL', 'CUSTAS PROCESSUAIS - CÍVEL ESTRATÉGICO', 'CUSTAS PROCESSUAIS - CÍVEL MASSA', 'CUSTAS PROCESSUAIS - IMOBILIÁRIO', 'CUSTAS PROCESSUAIS - REGULATÓRIO', 'CUSTAS PROCESSUAIS - TRABALHISTA', 'CUSTAS PROCESSUAIS - CÍVEL MASSA - CARTÕES', 'CUSTAS PROCESSUAIS - CÍVEL MASSA - FORNECEDOR', 'CUSTAS PROCESSUAIS - CÍVEL MASSA - MKTPLACE', 'CUSTAS PROCESSUAIS - CÍVEL MASSA - SEGURO', 'PAGAMENTO DE CUSTAS - TRIBUTÁRIO', 'PERITOS - REGULATÓRIO', 'RNO - CUSTAS PROCESSUAIS - TRABALHISTA')
        THEN VALOR
    END AS CUSTAS,
    CASE 
        WHEN `SUB TIPO` IN ('INSS - TRABALHISTA', 'IR - TRABALHISTA', 'PAGAMENTO DE INSS - INDENIZAÇÃO TRABALHISTA (FK)', 'PAGAMENTO DE IR - INDENIZAÇÃO TRABALHISTA (FK)', 'RNO - INSS - TRABALHISTA', 'RNO - IR - TRABALHISTA ', 'RNO - FGTS', 'FGTS')
        THEN VALOR
    END AS IMPOSTO,
    CASE 
        WHEN `SUB TIPO` IN ('LIBERAÇÃO DE PENHORA - MASSA', 'LIBERAÇÃO DE PENHORA MASSA', 'LIBERAÇÃO DE PENHORA - REGULATÓRIO')
        THEN VALOR
    END AS PENHORA,
    CASE 
        WHEN `SUB TIPO` IN ('PENSÃO - CÍVEL', 'PENSÃO - CÍVEL ESTRATÉGICO', 'PENSÃO - CÍVEL MASSA')
        THEN VALOR
    END AS PENSAO,
    CASE 
        WHEN `SUB TIPO` IN ('ACERTO CONTÁBIL: BLOQUEIO COM TRANSFERÊNCIA BANCÁRIA - DEP. JUDICIAL  (PAG)', 'ACERTO CONTÁBIL: BLOQUEIO COM TRANSFERÊNCIA BANCÁRIA - PAGAMENTO', 'ACERTO CONTÁBIL: PAGAMENTO DE ACORDO COM BAIXA DE DEP. JUDICIAL', 'LIMINAR (CÍV. MASSA)', 'PAGAMENTO', 'PAGAMENTO AUTUAÇÃO MINISTÉRIO DO TRABALHO - TRABALHISTA', 'PAGAMENTO DE EXECUÇÃO - TRABALHISTA', 'PAGAMENTOS - ALL - VV', 'PAGAMENTOS FK')
        THEN VALOR
    END AS OUTROS_PAGAMENTOS
FROM HISTORICO_DE_PAGAMENTOS_1
WHERE `SUB TIPO` IN ('ACORDO - CÍVEL', 'ACORDO - CÍVEL ESTRATÉGICO', 'ACORDO - CÍVEL MASSA', 'ACORDO - TRABALHISTA', 'PAGAMENTO DE ACORDO - TRABALHISTA (FK)', 'RNO - ACORDO - TRABALHISTA', 'ACORDO - CÍVEL MASSA - CARTÕES', 'ACORDO - CÍVEL MASSA - FORNECEDOR', 'ACORDO - CÍVEL MASSA - MKTPLACE', 'ACORDO - CÍVEL MASSA - SEGURO', 'ACORDO - IMOBILIÁRIO', 'ACORDO - MEDIAÇÃO', 'ACORDO - REGULATÓRIO - PROCON COM AUDIÊNCIA', 'ACORDO/ PROGRAMAS DE PARCELAMENTO - REGULATÓRIO', 'CONDENAÇÃO - CÍVEL', 'CONDENAÇÃO - CÍVEL ESTRATÉGICO', 'CONDENAÇÃO - CÍVEL MASSA', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO)', 'CONDENAÇÃO - REGULATÓRIO', 'CONDENAÇÃO - TRABALHISTA', 'CONDENAÇÃO - TRABALHISTA (INCONTROVERSO)', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO) - CARTÕES', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO) - FORNECEDOR', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO) - MKTPLACE', 'CONDENAÇÃO - CÍVEL MASSA (INCONTROVERSO) - SEGURO', 'CONDENAÇÃO - CÍVEL MASSA - CARTÕES', 'CONDENAÇÃO - CÍVEL MASSA - FORNECEDOR', 'CONDENAÇÃO - CÍVEL MASSA - MKTPLACE', 'CONDENAÇÃO - CÍVEL MASSA - SEGURO', 'CONDENAÇÃO - IMOBILIÁRIO', 'CONDENAÇÃO - IMOBILIÁRIO (INCONTROVERSO)', 'CONDENAÇÃO - REGULATÓRIO', 'MULTA - REGULATÓRIO', 'MULTA PROCESSUAL - CÍVEL', 'MULTA PROCESSUAL - TRABALHISTA', 'MULTA PROCESSUAL - CÍVEL ESTRATÉGICO', 'MULTA PROCESSUAL - IMOBILIÁRIO', 'RNO - CONDENAÇÃO - TRABALHISTA', 'RNO - CONDENAÇÃO - TRABALHISTA (INCONTROVERSO)', 'RNO - MULTA PROCESSUAL - TRABALHISTA', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA', 'HONORÁRIOS PERICIAIS - CÍVEL ESTRATÉGICO', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA', 'HONORÁRIOS PERICIAIS - IMOBILIÁRIO', 'HONORÁRIOS PERICIAIS - TRABALHISTA', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA - CARTÕES', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA - FORNECEDOR', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA - MKTPLACE', 'HONORÁRIOS CONCILIADOR - CIVEL MASSA - SEGURO', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA - CARTÕES', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA - FORNECEDOR', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA - MKTPLACE', 'HONORÁRIOS PERICIAIS - CÍVEL MASSA - SEGURO', 'HONORÁRIOS SUCUMBENCIAIS - REGULATÓRIO', 'PAGAMENTO DE HONORÁRIOS PERICIAIS - TRABALHISTA (FK)', 'RNO - HONORÁRIOS PERICIAIS -TRABALHISTA', 'CUSTAS FK - TRABALHISTA', 'CUSTAS PERITOS - CÍVEL', 'CUSTAS PERITOS - IMOBILIÁRIO', 'CUSTAS PROCESSUAIS - CÍVEL', 'CUSTAS PROCESSUAIS - CÍVEL ESTRATÉGICO', 'CUSTAS PROCESSUAIS - CÍVEL MASSA', 'CUSTAS PROCESSUAIS - IMOBILIÁRIO', 'CUSTAS PROCESSUAIS - REGULATÓRIO', 'CUSTAS PROCESSUAIS - TRABALHISTA', 'CUSTAS PROCESSUAIS - CÍVEL MASSA - CARTÕES', 'CUSTAS PROCESSUAIS - CÍVEL MASSA - FORNECEDOR', 'CUSTAS PROCESSUAIS - CÍVEL MASSA - MKTPLACE', 'CUSTAS PROCESSUAIS - CÍVEL MASSA - SEGURO', 'PAGAMENTO DE CUSTAS - TRIBUTÁRIO', 'PERITOS - REGULATÓRIO', 'RNO - CUSTAS PROCESSUAIS - TRABALHISTA', 'INSS - TRABALHISTA', 'IR - TRABALHISTA', 'PAGAMENTO DE INSS - INDENIZAÇÃO TRABALHISTA (FK)', 'PAGAMENTO DE IR - INDENIZAÇÃO TRABALHISTA (FK)', 'RNO - INSS - TRABALHISTA', 'RNO - IR - TRABALHISTA ', 'RNO - FGTS', 'FGTS', 'LIBERAÇÃO DE PENHORA - MASSA', 'LIBERAÇÃO DE PENHORA MASSA', 'LIBERAÇÃO DE PENHORA - REGULATÓRIO', 'PENSÃO - CÍVEL', 'PENSÃO - CÍVEL ESTRATÉGICO', 'PENSÃO - CÍVEL MASSA', 'ACERTO CONTÁBIL: BLOQUEIO COM TRANSFERÊNCIA BANCÁRIA - DEP. JUDICIAL  (PAG)', 'ACERTO CONTÁBIL: BLOQUEIO COM TRANSFERÊNCIA BANCÁRIA - PAGAMENTO', 'ACERTO CONTÁBIL: PAGAMENTO DE ACORDO COM BAIXA DE DEP. JUDICIAL', 'LIMINAR (CÍV. MASSA)', 'PAGAMENTO', 'PAGAMENTO AUTUAÇÃO MINISTÉRIO DO TRABALHO - TRABALHISTA', 'PAGAMENTO DE EXECUÇÃO - TRABALHISTA', 'PAGAMENTOS - ALL - VV', 'PAGAMENTOS FK')
    AND UPPER(`STATUS DO PAGAMENTO`) NOT IN ('REMOVIDO', 'CANCELADO', 'REJEITADO', 'EM CORREÇÃO', 'PENDENTE')
    AND UPPER(`STATUS DO PAGAMENTO`) NOT LIKE ('%PAGAMENTO DEVOLVIDO%')
""")

In [0]:
# Correção da DATA EFETIVA PAGAMENTO
df_pgto = df_pgto.withColumn('DATA EFETIVA DO PAGAMENTO', coalesce('DATA EFETIVA DO PAGAMENTO', 'DATA SOLICITAÇÃO (ESCRITÓRIO)'))


In [0]:
df_pgto = adjust_column_names(df_pgto)

In [0]:
df_pgto.createOrReplaceTempView("HISTORICO_DE_PAGAMENTOS_JUD_2")
# df_pgto.write.format("delta").mode("overwrite").saveAsTable(f"databox.juridico_comum.stg_historico_pgto")

In [0]:
df_pgto = spark.sql("""
/* AGRUPA TABELA DE PAGAMENTOS DE ACORDOS E CONDENAÇÕES */
WITH CTE AS (
    SELECT PROCESSO_ID
           ,DATA_EFETIVA_DO_PAGAMENTO
           ,COALESCE(SUM(ACORDO), 0) AS ACORDO
           ,COALESCE(SUM(CONDENACAO), 0) AS CONDENACAO
           ,COALESCE(SUM(PENHORA), 0) AS PENHORA
           ,COALESCE(SUM(PENSAO), 0) AS PENSAO
           ,COALESCE(SUM(OUTROS_PAGAMENTOS), 0) AS OUTROS_PAGAMENTOS
           ,COALESCE(SUM(IMPOSTO), 0) AS IMPOSTO
           --,COALESCE(SUM(CUSTAS), 0) AS CUSTAS
    FROM HISTORICO_DE_PAGAMENTOS_JUD_2
    GROUP BY 1, 2)
    SELECT CTE.*
    FROM CTE
    WHERE (ACORDO + CONDENACAO + PENHORA + OUTROS_PAGAMENTOS + IMPOSTO) > 0 -- + PENSAO + CUSTAS
"""
)

In [0]:
# display(df_pgto.sort("PROCESSO_ID", ascending=True))

In [0]:
df_pgto.count() #575093

606175

### 3- Tratamento base de Garantias


In [0]:
# Ajusta os nomes das colunas com '_' no lugar de ' '
df_garantias = adjust_column_names(df_garantias)

In [0]:
df_garantias.createOrReplaceTempView("TB_GARANTIAS_F")

In [0]:
df_garantias.createOrReplaceTempView("TB_GARANTIAS_MES_A")

In [0]:
df_garantias = spark.sql("""
SELECT PROCESSO_ID
        ,`DATA_DE_LEVANTAMENTO_PARTE_CONTRÁRIA`
		,`VALOR_LEVANTADO_PARTE_CONTRÁRIA`
FROM TB_GARANTIAS_MES_A
WHERE STATUS_DA_GARANTIA <> 'CANCELADO'
		AND STATUS_DA_GARANTIA NOT LIKE ('%PAGAMENTO DEVOLVIDO%')
		AND STATUS <> 'REMOVIDO'
		AND TIPO_GARANTIA IN ('ACERTO CONTÁBIL: BLOQUEIO COM TRANSFERÊNCIA BANCÁRIA - DEP. JUDICIAL',
						'DEPÓSITO',
						'DEPÓSITO GARANTIA DE JUIZO - CÍVEL',
						'DEPÓSITO JUDICIAL',
						'DEPÓSITO JUDICIAL - CÍVEL',
						'DEPÓSITO JUDICIAL - CÍVEL ESTRATÉGICO',
						'DEPÓSITO JUDICIAL - CÍVEL MASSA',
						'DEPÓSITO JUDICIAL - REGULATÓRIO',
						'DEPÓSITO JUDICAL REGULATÓRIO',
						'DEPÓSITO JUDICIAL - TRABALHISTA',
						'DEPOSITO JUDICIAL TRABALHISTA',
						'DEPÓSITO JUDICIAL TRABALHISTA - BLOQUEIO',
						'DEPÓSITO JUDICIAL TRABALHISTA BLOQUEIO - TRABALHISTA',
						'DEPÓSITO JUDICIAL TRIBUTÁRIO',
						'DEPÓSITO RECURSAL - AIRO - TRABALHISTA',
						'DEPÓSITO RECURSAL - AIRR',
						'DEPÓSITO RECURSAL - AIRR - TRABALHISTA',
						'DEPÓSITO RECURSAL - CÍVEL MASSA',
						'DEPÓSITO RECURSAL - EMBARGOS TST',
						'DEPÓSITO RECURSAL - RO - TRABALHISTA',
						'DEPÓSITO RECURSAL - RR - TRABALHISTA',
						'DEPÓSITO RECURSAL AIRR',
						'DEPÓSITO RECURSAL RO',
						'PENHORA - GARANTIA',
						'PENHORA - REGULATÓRIO' );
""")
# df_garantias.count() #75479

In [0]:
df_garantias.createOrReplaceTempView("TB_GARANTIAS_F")

df_garantias = spark.sql("""
	SELECT PROCESSO_ID
        ,`DATA_DE_LEVANTAMENTO_PARTE_CONTRÁRIA` AS DATA_EFETIVA_DO_PAGAMENTO
        ,SUM(`VALOR_LEVANTADO_PARTE_CONTRÁRIA`) AS GARANTIA
	FROM TB_GARANTIAS_F
  WHERE `DATA_DE_LEVANTAMENTO_PARTE_CONTRÁRIA` IS NOT NULL
	GROUP BY 1, 2
	ORDER BY 1, 2
 """)

In [0]:
df_garantias.count() # 41244 linhas

44219

### 4 - Junta as bases de Pagamentos e Garantias

In [0]:
df_pgto.createOrReplaceTempView("HISTORICO_DE_PAGAMENTOS_JUD_4")
df_garantias.createOrReplaceTempView("TB_GARANTIAS_F")

In [0]:
# Faz o Join das duas tabelas
df_pgto_garantias = spark.sql("""
SELECT 
coalesce(A.PROCESSO_ID, B.PROCESSO_ID) AS PROCESSO_ID
,coalesce(A.DATA_EFETIVA_DO_PAGAMENTO, B.DATA_EFETIVA_DO_PAGAMENTO) AS DATA_EFETIVA_DO_PAGAMENTO
,ACORDO
,CONDENACAO
,PENHORA
,PENSAO
,OUTROS_PAGAMENTOS
,IMPOSTO
--,CUSTAS
,coalesce(GARANTIA, 0) AS GARANTIA
FROM HISTORICO_DE_PAGAMENTOS_JUD_4 A
FULL OUTER JOIN TB_GARANTIAS_F B
ON A.PROCESSO_ID = B.PROCESSO_ID AND A.DATA_EFETIVA_DO_PAGAMENTO = B.DATA_EFETIVA_DO_PAGAMENTO;
""")

In [0]:
df_pgto_garantias = df_pgto_garantias.withColumn("TOTAL_PAGAMENTOS",col("ACORDO") + 
                                                                    col("CONDENACAO") +
                                                                    col("PENHORA") +
                                                                    col("PENSAO") +
                                                                    col("OUTROS_PAGAMENTOS") +
                                                                    col("IMPOSTO") +
                                                                    col("GARANTIA"))

In [0]:
df_pgto_garantias.createOrReplaceGlobalTempView("HIST_PAGAMENTOS_GARANTIA_TRAB_F")

In [0]:
# Agrupa as infromações
df_pgto_garantias = spark.sql("""
    SELECT PROCESSO_ID
        ,MAX(DATA_EFETIVA_DO_PAGAMENTO) AS DT_ULT_PGTO
        ,SUM(coalesce(ACORDO, 0)) AS ACORDO
        ,SUM(coalesce(CONDENACAO, 0)) AS CONDENACAO
        ,SUM(coalesce(PENHORA, 0)) AS PENHORA
        ,SUM(coalesce(PENSAO, 0)) AS PENSAO
        ,SUM(coalesce(OUTROS_PAGAMENTOS, 0)) AS OUTROS_PAGAMENTOS
        ,SUM(coalesce(IMPOSTO, 0)) AS IMPOSTO
        ,SUM(coalesce(GARANTIA, 0)) AS GARANTIA
    FROM global_temp.HIST_PAGAMENTOS_GARANTIA_TRAB_F
    GROUP BY 1 
    ORDER BY 1;
 """)

In [0]:
df_pgto_garantias.createOrReplaceGlobalTempView('HIST_PAGAMENTOS_GARANTIA_TRAB_FF')
# df_pgto_garantias.write.format("delta").mode("overwrite").saveAsTable(f"databox.juridico_comum.stg_trab_pgto_garantias")

In [0]:
dbutils.notebook.exit("Success")